# MEDS Meta File Review

In this python notebook I will look at one meta file for each float type Argo Canada operates and suggest changes/additions to the fields based on how we are operating each float. 

In [2]:
# imports

import sys
from netCDF4 import Dataset
import pandas as pd

import bgcArgoDMQC as bgc
import argopy

In [3]:
ix = argopy.ArgoIndex().to_dataframe()
meds = ix.loc[ix.dac == 'meds']

In [4]:
profilers = meds.profiler.unique()
wmo = [meds.loc[meds.profiler == p, 'wmo'].iloc[-1] for p in profilers]
profilers = pd.DataFrame(dict(wmo=wmo, dac=len(wmo)*['meds'], profiler=profilers))
profilers = pd.concat((profilers, pd.DataFrame(dict(wmo=[7901133], dac=['coriolis'], profiler=[ix.loc[ix.wmo == 7901133, 'profiler'].iloc[0]]))))
profilers

,wmo,dac,profiler
0,4901175,meds,Teledyne Webb Research float with SBE conducti...
1,4900539,meds,PROVOR float with SBE conductivity sensor
2,4902466,meds,NOVA float with SBE conductivity sensor
3,4901771,meds,POPS ice platform using PROVOR float with SBE ...
4,4902704,meds,ARVOR float with SBE conductivity sensor
5,4902671,meds,ARVOR float with RBR conductivity sensor
6,4902683,meds,PROVOR III
7,4902647,meds,ARVOR-D deep float with SBE conductivity sensor
8,4902691,meds,PROVOR V SBE
0,7901133,coriolis,PROVOR V SBE


In [5]:
for row in profilers.iterrows():
    wmo = row[1].wmo
    prof = row[1].profiler
    dac = row[1].dac
    bgc.io.get_argo(wmo, local_path='data/dac', ftype='summary', summary_overwrite=False)
    meta_file = f'data/dac/{dac}/{wmo}/{wmo}_meta.nc'
    meta = Dataset(meta_file)
    with open(f'data/text/{wmo}_meta_text.txt', 'w') as f:
        f.write(f'\n{prof}\n\n')
        for k in meta.variables.keys():
            f.write(repr(meta[k]))
            f.write('\n\n')
            try:
                if len(meta[k].dimensions) > 1:
                    for row in meta[k][:].data:
                        f.write(f'{bgc.io.read_ncstr(row)}\n')
                else:
                    f.write(f'{bgc.io.read_ncstr(meta[k][:].data)}\n')
            except:
                f.write('')
            f.write('\n')
        meta.close()

In [7]:
meta = Dataset(meta_file)
for name, value in zip(meta['LAUNCH_CONFIG_PARAMETER_NAME'][:].data, meta['LAUNCH_CONFIG_PARAMETER_VALUE'][:].data):
    print(f'{bgc.io.read_ncstr(name)}', value)

CONFIG_TelemetryAutoTestAtDeployment_LOGICAL 1.0
CONFIG_PressureOffsetSampledAfterTransmission_LOGICAL 1.0
CONFIG_DescentSpeedMin_cm/s 1.0
CONFIG_AscentSpeedMin_cm/s 8.33
CONFIG_PressureCheckTimeDescent_seconds 300.0
CONFIG_PressureCheckTimeAscent_seconds 120.0
CONFIG_PressureCheckTimeParking_seconds 1800.0
CONFIG_OilVolumeMaxPerValveAction_cm^3 20.0
CONFIG_OilVolumeMaxPerPumpActionReposition_cm^3 20.0
CONFIG_BuoyancyReductionFirstThreshold_dbar 0.8
CONFIG_BuoyancyReductionSecondThreshold_dbar 1.5
CONFIG_OilVolumePerPumpActionAscent_cm^3 15.0
CONFIG_ValveTimeActionBuoyancyReduction_csec 2700.0
CONFIG_ValveActionPeriodFactorBuoyancyReduction_NUMBER 2.23
CONFIG_AscentEndThreshold_dbar 5.0
CONFIG_GPSSetClockSensorCard_LOGICAL 0.0
CONFIG_CtdDepthZone1DataSynchronizationMode_NUMBER 0.0
CONFIG_CtdDepthZone2DataSynchronizationMode_NUMBER 0.0
CONFIG_CtdDepthZone3DataSynchronizationMode_NUMBER 0.0
CONFIG_CtdDepthZone4DataSynchronizationMode_NUMBER 0.0
CONFIG_CtdDepthZone5DataSynchronizationMode